In [6]:
import pandas as pd
import numpy as np
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import pylandstats as pls
from exactextract import exact_extract

In [2]:
# aachen_bbox = gpd.read_file('../../data/processed/Aachen_67/bbox.geojson')
aachen_lst_path = '../../data/processed/Aachen_67/67_lst.tif'
aachen_building_height_path = '../../data/processed/Aachen_67/67_building_heights_2018.tif'
aachen_built_up_path = '../../data/processed/Aachen_67/67_built_up.tif'
aachen_gaia_path = '../../data/processed/Aachen_67/67_gaia_2018.tif'
aachen_esa_path = '../../data/processed/Aachen_67/67_worldcover2021.tif'
aachen_500m_grid = gpd.read_parquet('../../data/processed/Aachen_67/grid_500m.parquet')

In [ ]:
# read teh lst data and print some stats about it
with rasterio.open(aachen_lst_path, 'r') as src:
    lst_data = src.read(1)
    current_nodata = src.nodata
    
    print(f'Current NoData value in metadata: {current_nodata}')
    print(f'LST data matrix shape: {lst_data.shape}')
    
    # 1. Create a secure mask to isolate valid ecological observations
    # Exclude both the metadata-defined NoData value and IEEE NaN values
    valid_mask = (~np.isnan(lst_data))
    if current_nodata is not None:
        valid_mask &= (lst_data != current_nodata)
        
    # 2. Extract valid pixels as a clean 1D vector
    valid_pixels = lst_data[valid_mask]
    
    # 3. Defensive check: ensure the image is not completely empty
    if valid_pixels.size > 0:
        print(f'Total valid pixels: {valid_pixels.size} ({valid_pixels.size / lst_data.size * 100:.2f}%)')
        print(f'LST valid data min: {np.min(valid_pixels):.4f}')
        print(f'LST valid data max: {np.max(valid_pixels):.4f}')
        print(f'LST valid data mean: {np.mean(valid_pixels):.4f}')
        print(f'LST valid data std: {np.std(valid_pixels):.4f}') # added standard deviation for completeness
        print(f'crs of the raster: {src.crs}')
        print("Raster Bounds:", src.bounds)
        print("Grid Bounds:  ", aachen_500m_grid.total_bounds)
        print("Grid CRS:     ", aachen_500m_grid.crs)
    else:
        print('Warning: The raster image contains NO valid pixel data.')

In [ ]:
def compute_zonal_mean_absolute(raster_path: str, grid_gdf: gpd.GeoDataFrame, output_col: str = 'raster_mean') -> gpd.GeoDataFrame:
    # 1. Inspect CRS alignment (They are confirmed aligned, but we keep this for multi-city safety)
    with rasterio.open(raster_path, 'r') as src:
        raster_crs = src.crs
        
        if grid_gdf.crs != raster_crs:
            processing_gdf = grid_gdf.to_crs(raster_crs)
        else:
            processing_gdf = grid_gdf.copy()
            
    # 2. Run exactextract explicitly wrapped as a single-element list to enforce standard output
    print(f"Extracting zonal mean for {len(processing_gdf)} geometries...")
    stats_df = exact_extract(raster_path, processing_gdf, 'mean', output='pandas')
        
    # 3. rename the output column to the user-specified name
    stats_df.rename(columns={'mean': output_col}, inplace=True)
    
    print("Pandas-accelerated zonal pipeline completed successfully.")
    return stats_df

In [ ]:
output_gdf = compute_zonal_mean_absolute(aachen_lst_path, aachen_500m_grid, output_col='lst_mean')

In [ ]:
output_gdf.head(20)

In [4]:
def cal_landstats(grid_gdf, landscape_raster_path):
    # calculate the landstats
    za = pls.ZonalAnalysis(landscape_raster_path, grid_gdf)

    # assgin the name of class level caculater and landscape level caculater
    class_level_cal = ['proportion_of_landscape', 'patch_density', 'edge_density', 'largest_patch_index', 'landscape_shape_index']
    landscape_level_cal = ['shannon_diversity_index', 'contagion']

    # calculate the class level landstats and save them into different df
    class_level_cal_df = za.compute_class_metrics_df(metrics=class_level_cal)
    landscape_level_cal_df = za.compute_landscape_metrics_df(metrics=landscape_level_cal)

    # reame the columns for both df 
    class_metrics_rename = class_level_cal_df.rename(columns={'proportion_of_landscape': 'PLAND', 'patch_density': 'PD', 'edge_density': 'ED', 'largest_patch_index': 'LPI', 'landscape_shape_index': 'LSI'})
    landscape_metrics_rename = landscape_level_cal_df.rename(columns={'shannon_diversity_index': 'SHDI', 'contagion': 'CONTAG'})

    # reset the index of both df 
    class_metrics_rename.reset_index(inplace=True)
    landscape_metrics_rename.reset_index(inplace=True)

    # pivot and rename the both df
    class_attributes = ['PLAND', 'PD', 'ED', 'LPI', 'LSI']
    class_pivoted_dfs = []
    for attr in class_attributes:
        class_pivot = class_metrics_rename.pivot_table(
            index='zone', 
            columns='class_val', 
            values=attr, 
            aggfunc='first'
        )
        class_pivot.columns = [f'{attr}_cls_{col}' for col in class_pivot.columns]
        class_pivoted_dfs.append(class_pivot)

    # Combine all pivoted tables into one
    class_pivot_results = pd.concat(class_pivoted_dfs, axis=1).reset_index()

    # join the two dataframes with the zone column
    landstats_df = pd.merge(class_pivot_results, landscape_metrics_rename, on='zone')

    # drop the zone column
    landstats_df.drop(columns='zone', inplace=True)

    return landstats_df

In [7]:
df_landstats = cal_landstats(aachen_500m_grid, aachen_esa_path)

[                                        ] | 0% Completed | 342.40 us

/opt/conda/envs/gee/lib/python3.13/site-packages/pyogrio/raw.py:200: RuntimeWarning: Invalid range specified 
  return ogr_read(


[########################################] | 100% Completed | 6.48 ss
[########################################] | 100% Completed | 1.73 sms


In [ ]:
# vis the data
fig, ax = plt.subplots(figsize=(10, 10))
aachen_bbox.plot(ax=ax, edgecolor='red', facecolor='none', linewidth=2, label='Bounding Box')
aachen_500m_grid.plot(ax=ax, edgecolor='blue', facecolor='none', linewidth=1, label='500m Grid')
plt.legend()